In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    _p = _root / "paths.py"
    if _p.exists() and "GPT4O_ROOT" in _p.read_text(encoding="utf-8", errors="ignore"):
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find GPT4o/paths.py — set Jupyter cwd to After_PT_Removal/GPT4o or a subfolder."
    )
import paths


In [ ]:
import openai
import pandas as pd

# Load the CSV file
df = pd.read_csv(paths.RAW / "GPT4o_Both_Rounds.csv")

# Display the first few rows after removing duplicates
# print(df_op4.columns)
print(df.head())
print(len(df))

   Origin       q1           q2       q3       q4       q5           q6  \
0  ID0002        1  0.833333333  REMOVED        1  REMOVED  0.833333333   
1  ID0003        1            1  REMOVED        1        1            1   
2  ID0007  REMOVED            1        1        1        1      REMOVED   
3  ID0009  REMOVED            1        1        1        1            1   
4  ID0010        1      REMOVED  REMOVED  REMOVED  REMOVED      REMOVED   

        q7       q8       q9  ...  \
0  REMOVED  REMOVED  REMOVED  ...   
1  REMOVED      NaN      NaN  ...   
2        1      NaN      NaN  ...   
3  REMOVED  REMOVED      NaN  ...   
4        1      NaN      NaN  ...   

                                    question_options  \
0  What Would You Do Next?\n\nA: Perform left upp...   
1  Which of the following components is essential...   
2  Which of the following diagnostic tests would ...   
3  What Is Your Diagnosis?\n\nA: Blue rubber bleb...   
4  What is the proper method for transporting 

In [2]:
import openai

def generate_direct_prediction(context, question):
    """
    Queries GPT-4o with a clinical vignette (context) and a multiple-choice question (with embedded options).
    Returns only the predicted answer in the format: '[Letter]: [Answer Text]' (e.g., 'B: Femoral artery murmur').
    """
    prompt = f"""
You are given some context and a multiple-choice question.

Select the most appropriate answer from the options provided.

{context}

{question}

Provide your response in the following format:\n<answer>Option [letter]</answer>"""

    try:
        client = openai.OpenAI()

        response = client.chat.completions.create(
            model="gpt-5",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except Exception:
        return "Error"


In [3]:
import pandas as pd
from tqdm import tqdm
import os

output_path = paths.GPT5_PREDICTIONS / "gpt5_predictions_on_gpt4o_removed.csv"

# If continuing from a previous batch, load the existing file and get already-completed indices
if os.path.exists(output_path):
    df_existing = pd.read_csv(output_path)
    completed_ids = set(df_existing.index)
    print(f"✅ Loaded existing file with {len(completed_ids)} completed rows.")
else:
    df_existing = pd.DataFrame()
    completed_ids = set()

# Collect new results in a list of dicts
new_rows = []

# Iterate with progress bar
for idx, row in tqdm(df.iterrows(), total=len(df)):
    if idx in completed_ids:
        continue  # skip already processed

    pred = generate_direct_prediction(row["New_Sentences"], row["question_options"])
    
    result_row = row.to_dict()
    result_row["gpt4o_direct_prediction"] = pred
    new_rows.append(result_row)

    # Write out after each row to ensure persistence
    df_batch = pd.DataFrame(new_rows)
    df_combined = pd.concat([df_existing, df_batch], ignore_index=True)
    df_combined.to_csv(output_path, index=False)


✅ Loaded existing file with 1300 completed rows.


100%|█████████████████████████████████████████████████| 1300/1300 [00:00<00:00, 36595.09it/s]


In [5]:
import pandas as pd
import re
import numpy as np

# Load the model predictions
output_path = paths.GPT5_PREDICTIONS / "gpt5_predictions_on_gpt4o_removed.csv"
df = pd.read_csv(output_path)

# Extract the predicted letter from the format <answer>Option A</answer>
def extract_letter_from_xml(pred):
    if isinstance(pred, str):
        match = re.search(r"Option\s+([A-J])", pred, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    return None

df["gpt_letter"] = df["gpt4o_direct_prediction"].apply(extract_letter_from_xml)

# Clean and standardize the ground truth answer
df["answer_letter"] = df["answer_corr"].astype(str).str.strip().str.upper()

# Compare predictions to ground truth
df["gpt_letter_match"] = df.apply(
    lambda row: "Correct" if row["gpt_letter"] == row["answer_letter"] else "Incorrect",
    axis=1
)

# Convert to binary for std calculation
df["gpt_letter_binary"] = df["gpt_letter_match"].map({"Correct": 1, "Incorrect": 0})

# Compute overall accuracy
correct_count = df["gpt_letter_binary"].sum()
total_count = df["gpt_letter_binary"].notna().sum()
accuracy = correct_count / total_count if total_count > 0 else 0

print(f"Letter-Based Correct Predictions: {correct_count}")
print(f"Total Predictions Compared: {total_count}")
print(f"Letter Match Accuracy: {accuracy:.2%}")

# Per-data source accuracy and std
for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = source_df["gpt_letter_binary"].sum()
    total = source_df["gpt_letter_binary"].notna().sum()
    acc = correct / total if total > 0 else 0
    std = source_df["gpt_letter_binary"].std(ddof=1) if total > 1 else float("nan")
    
    print(f"Data Source: {source}")
    print(f"  Correct Predictions: {correct}")
    print(f"  Total Predictions: {total}")
    print(f"  Accuracy: {acc:.2%}")
    print(f"  Std Dev: {std:.4f}\n")


Letter-Based Correct Predictions: 1031
Total Predictions Compared: 1300
Letter Match Accuracy: 79.31%
Data Source: jama
  Correct Predictions: 496
  Total Predictions: 582
  Accuracy: 85.22%
  Std Dev: 0.3552

Data Source: medxpert
  Correct Predictions: 200
  Total Predictions: 318
  Accuracy: 62.89%
  Std Dev: 0.4839

Data Source: medbullets
  Correct Predictions: 173
  Total Predictions: 207
  Accuracy: 83.57%
  Std Dev: 0.3714

Data Source: mmlu
  Correct Predictions: 162
  Total Predictions: 193
  Accuracy: 83.94%
  Std Dev: 0.3681



In [6]:
import numpy as np

# Collect accuracies per data source
accuracies = []

for source in df["data_source_corr"].unique():
    source_df = df[df["data_source_corr"] == source]
    correct = (source_df["gpt_letter_match"] == "Correct").sum()
    total = source_df["gpt_letter_match"].notna().sum()
    acc = correct / total if total > 0 else 0
    accuracies.append(acc)

# Calculate standard deviation
accuracy_std = np.std(accuracies, ddof=1)  # use ddof=1 for sample std deviation
print(f"Standard Deviation of Accuracy Across Data Sources: {accuracy_std:.4f}")


Standard Deviation of Accuracy Across Data Sources: 0.1070
